# DeepACO 实验 notebook

本 notebook 覆盖三组实验（请在 `tsp` 目录下打开，保证 `COMPARE_LOG_DIR` 与导入路径正确）：

1. **实验一**：每个规模两个单元格——仅 baseline（`ucb_c=0`，日志 `compare_{preset}_baseline.log`），仅改进 schedule（日志 `compare_{preset}_schedule.log`，追加写入并带时间戳横幅）。tsp500 按 `[::8]` 抽样。tsp100 全量。
2. **实验二**：单元格内**内联** [tune_tsp20.py](tune_tsp20.py) 全文逻辑（非 `import`）；函数名 `tune20_` 前缀以免覆盖实验一；日志 `tune_tsp20.log`。
3. **实验三**：单元格内**内联** [tune_tsp20_ablation_decay_only.py](tune_tsp20_ablation_decay_only.py) 逻辑（`abl20_` 前缀）；可复用实验二的 `net_tune`/`tune_test_list`；日志 `tune_tsp20_ablation_decay_only.log`。

模型权重：`../pretrained/tsp/tsp{n_node}.pt`；测试集：`../data/tsp/testDataset-{n_node}.pt`。


In [2]:
import sys
import time
import itertools
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime

import torch

from net import Net
from aco import ACO
from utils import load_test_dataset

torch.manual_seed(12345)

EPS = 1e-10
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'device={device}')



libgomp: Invalid value for environment variable OMP_NUM_THREADS


device=cuda:0


In [3]:
@torch.no_grad()
def infer_instance(
    model,
    pyg_data,
    distances,
    n_ants,
    t_aco_diff,
    k_sparse=None,
    ucb_c: float = 0.0,
    ucb_schedule=None,
    ucb_c_min: float = 0.0,
    ucb_tau=None,
    stagnation_k: int = 0,
    boost_gamma: float = 2.0,
    ucb_c_max=None,
    boost_cooldown: int = 0,
):
    if model is not None:
        model.eval()
        heu_vec = model(pyg_data)
        heu_mat = model.reshape(pyg_data, heu_vec) + EPS
        aco = ACO(
            n_ants=n_ants,
            heuristic=heu_mat,
            distances=distances,
            device=device,
            ucb_c=ucb_c,
            ucb_schedule=ucb_schedule,
            ucb_c_min=ucb_c_min,
            ucb_tau=ucb_tau,
            stagnation_k=stagnation_k,
            boost_gamma=boost_gamma,
            ucb_c_max=ucb_c_max,
            boost_cooldown=boost_cooldown,
        )
    else:
        aco = ACO(
            n_ants=n_ants,
            distances=distances,
            device=device,
            ucb_c=ucb_c,
            ucb_schedule=ucb_schedule,
            ucb_c_min=ucb_c_min,
            ucb_tau=ucb_tau,
            stagnation_k=stagnation_k,
            boost_gamma=boost_gamma,
            ucb_c_max=ucb_c_max,
            boost_cooldown=boost_cooldown,
        )
        if k_sparse:
            aco.sparsify(k_sparse)

    results = torch.zeros(size=(len(t_aco_diff),), device=device)
    for i, t in enumerate(t_aco_diff):
        best_cost = aco.run(t)
        results[i] = best_cost
    return results


@torch.no_grad()
def test(dataset, model, n_ants, t_aco, k_sparse=None, **kwargs):
    _t_aco = [0] + list(t_aco)
    t_aco_diff = [_t_aco[i + 1] - _t_aco[i] for i in range(len(_t_aco) - 1)]
    sum_results = torch.zeros(size=(len(t_aco_diff),), device=device)
    start = time.time()
    for pyg_data, distances in dataset:
        sum_results += infer_instance(
            model, pyg_data, distances, n_ants, t_aco_diff, k_sparse, **kwargs
        )
    end = time.time()
    return sum_results / len(dataset), end - start


In [16]:
N_ANTS = 20
T_ACO = [1, 10, 20, 30, 40, 50, 60, 70, 80 ,90 , 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]

COMPARE_LOG_DIR = Path.cwd()

presets = {
    'tsp20': {
        'n_node': 20,
        'k_sparse': 10,
        'schedule': dict(
            ucb_c=0.8,
            ucb_schedule='exp',
            ucb_c_min=0.05,
            ucb_tau=20.0,
            stagnation_k=10,
            boost_gamma=2.0,
            ucb_c_max=2.0,
            boost_cooldown=5,
        ),
    },
    'tsp500': {
        'n_node': 500,
        'k_sparse': 50,
        # 大规模问题沿用 tsp100 的 "no boost · decay only" 配置
        'schedule': dict(
            ucb_c=2,
            ucb_schedule='exp',
            ucb_c_min=0.0,
            ucb_tau=10.0,
            stagnation_k=10,
            boost_gamma=5.0,
            ucb_c_max=None,
            boost_cooldown=10,
        ),
    },
    'tsp100': {
        'n_node': 100,
        'k_sparse': 20,
        'schedule': dict(
            ucb_c=1.4,
            ucb_schedule='exp',
            ucb_c_min=0.0,
            ucb_tau=10.0,
            stagnation_k=10,
            boost_gamma=5.0,
            ucb_c_max=None,
            boost_cooldown=10,
        ),
    },
}

baseline_kwargs = dict(ucb_c=0.0)


class _Tee:
    """将 print 同时写到控制台与 log 文件（每次 write 后 flush）。"""

    __slots__ = ('_streams',)

    def __init__(self, *streams):
        self._streams = streams

    def write(self, data: str):
        for s in self._streams:
            s.write(data)
            s.flush()
        return len(data)

    def flush(self):
        for s in self._streams:
            s.flush()


@contextmanager
def tee_stdout(log_path, mode='w'):
    log_path = Path(log_path)
    with open(log_path, mode, encoding='utf-8') as log_f:
        old_out = sys.stdout
        sys.stdout = _Tee(old_out, log_f)
        try:
            yield
        finally:
            sys.stdout = old_out


def load_model_for(n_node):
    net = Net().to(device)
    net.load_state_dict(
        torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device)
    )
    return net


def _print_costs(prefix, t_aco, avg, dur):
    print(f'{prefix} duration={dur:.2f}s')
    for i, t in enumerate(t_aco):
        print(f'  T={t:>3}  avg_cost={float(avg[i]):.6f}')


def run_baseline_only(preset_key, subsample_step: int = 1):
    """仅 baseline（ucb_c=0）。subsample_step>1 时按 dataset[::step] 抽样。"""
    cfg = presets[preset_key]
    n_node, k_sparse = cfg['n_node'], cfg['k_sparse']

    full_list = load_test_dataset(n_node, k_sparse, device)
    test_list = full_list[::subsample_step] if subsample_step > 1 else full_list
    net = load_model_for(n_node)
    print(f'== {preset_key} baseline == full={len(full_list)} '
          f'subset(step={subsample_step})={len(test_list)} k_sparse={k_sparse} '
          f'n_ants={N_ANTS} t_aco={T_ACO}')

    avg_b, dur_b = test(test_list, net, N_ANTS, T_ACO, k_sparse, **baseline_kwargs)
    print()
    _print_costs('[baseline c=0]', T_ACO, avg_b, dur_b)
    return dict(avg=avg_b, dur=dur_b, preset_key=preset_key, subsample_step=subsample_step)


def run_schedule_only(preset_key, subsample_step: int = 1):
    """仅改进 schedule（presets 中的 schedule 字典）。"""
    cfg = presets[preset_key]
    n_node, k_sparse = cfg['n_node'], cfg['k_sparse']
    schedule_kwargs = cfg['schedule']

    full_list = load_test_dataset(n_node, k_sparse, device)
    test_list = full_list[::subsample_step] if subsample_step > 1 else full_list
    net = load_model_for(n_node)
    print(f'== {preset_key} schedule == full={len(full_list)} '
          f'subset(step={subsample_step})={len(test_list)} k_sparse={k_sparse} '
          f'n_ants={N_ANTS} t_aco={T_ACO}')

    avg_s, dur_s = test(test_list, net, N_ANTS, T_ACO, k_sparse, **schedule_kwargs)
    print(f'\n[schedule cfg] {schedule_kwargs}')
    _print_costs('[schedule]', T_ACO, avg_s, dur_s)
    return dict(
        avg=avg_s,
        dur=dur_s,
        schedule_kwargs=schedule_kwargs,
        preset_key=preset_key,
        subsample_step=subsample_step,
    )


def print_schedule_run_banner(preset_key):
    """schedule 日志追加多次运行时打印分隔与时间戳。"""
    print(
        f"\n{'=' * 60}\n"
        f"{datetime.now().isoformat()}  {preset_key}\n"
        f"schedule={presets[preset_key]['schedule']}\n"
        f"{'=' * 60}\n"
    )


def print_delta(
    t_aco,
    avg_b,
    avg_s,
    label='[delta vs baseline] (negative = schedule better)',
):
    print(label)
    for i, t in enumerate(t_aco):
        b, s = float(avg_b[i]), float(avg_s[i])
        delta = s - b
        pct = 100.0 * delta / b if b != 0 else float('nan')
        print(f'  T={t:>3}  delta={delta:+.6f}  ({pct:+.3f}%)')


## 实验一：完整测试集对比（每规模两格）

每规模 **baseline** 与 **schedule** 分属不同单元格：baseline 只需偶尔运行；调参时可反复单独运行 schedule 单元格。schedule 日志为追加写入（`append`），每次运行前自动打印时间戳与当前 `schedule` 配置。

可选：三组均跑完后，运行下方的 **delta** 单元格查看差值。


In [8]:
with tee_stdout(COMPARE_LOG_DIR / 'compare_tsp20_baseline.log', 'w'):
    res_bl_tsp20 = run_baseline_only('tsp20')


== tsp20 baseline == full=1280 subset(step=1)=1280 k_sparse=10 n_ants=20 t_aco=[1, 10, 20, 30, 40, 50, 100]

[baseline c=0] duration=1544.55s
  T=  1  avg_cost=3.968287
  T= 10  avg_cost=3.863602
  T= 20  avg_cost=3.855592
  T= 30  avg_cost=3.853167
  T= 40  avg_cost=3.851936
  T= 50  avg_cost=3.851151
  T=100  avg_cost=3.848487


In [9]:
with tee_stdout(COMPARE_LOG_DIR / 'compare_tsp20_schedule.log', 'a'):
    print_schedule_run_banner('tsp20')
    res_sc_tsp20 = run_schedule_only('tsp20')



2026-05-04T09:40:49.099120  tsp20
schedule={'ucb_c': 0.8, 'ucb_schedule': 'exp', 'ucb_c_min': 0.05, 'ucb_tau': 20.0, 'stagnation_k': 10, 'boost_gamma': 2.0, 'ucb_c_max': 2.0, 'boost_cooldown': 5}

== tsp20 schedule == full=1280 subset(step=1)=1280 k_sparse=10 n_ants=20 t_aco=[1, 10, 20, 30, 40, 50, 100]

[schedule cfg] {'ucb_c': 0.8, 'ucb_schedule': 'exp', 'ucb_c_min': 0.05, 'ucb_tau': 20.0, 'stagnation_k': 10, 'boost_gamma': 2.0, 'ucb_c_max': 2.0, 'boost_cooldown': 5}
[schedule] duration=1544.41s
  T=  1  avg_cost=3.972020
  T= 10  avg_cost=3.864520
  T= 20  avg_cost=3.856413
  T= 30  avg_cost=3.853604
  T= 40  avg_cost=3.851844
  T= 50  avg_cost=3.850435
  T=100  avg_cost=3.847807


In [10]:
with tee_stdout(COMPARE_LOG_DIR / 'compare_tsp500_baseline.log', 'w'):
    res_bl_tsp500 = run_baseline_only('tsp500', subsample_step=8)


== tsp500 baseline == full=128 subset(step=8)=16 k_sparse=50 n_ants=20 t_aco=[1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]

[baseline c=0] duration=734.12s
  T=  1  avg_cost=20.961203
  T= 10  avg_cost=19.673380
  T= 20  avg_cost=19.423306
  T= 30  avg_cost=19.240623
  T= 40  avg_cost=19.044125
  T= 50  avg_cost=18.984301
  T= 60  avg_cost=18.883335
  T= 70  avg_cost=18.848593
  T= 80  avg_cost=18.836060
  T= 90  avg_cost=18.788584
  T=100  avg_cost=18.766872
  T=110  avg_cost=18.760654
  T=120  avg_cost=18.741608
  T=130  avg_cost=18.728489
  T=140  avg_cost=18.713387
  T=150  avg_cost=18.694096
  T=160  avg_cost=18.694096
  T=170  avg_cost=18.693224
  T=180  avg_cost=18.693224
  T=190  avg_cost=18.693224
  T=200  avg_cost=18.693224


In [17]:
with tee_stdout(COMPARE_LOG_DIR / 'compare_tsp500_schedule.log', 'a'):
    print_schedule_run_banner('tsp500')
    res_sc_tsp500 = run_schedule_only('tsp500', subsample_step=8)


2026-05-05T11:02:20.423793  tsp500
schedule={'ucb_c': 1.6, 'ucb_schedule': 'exp', 'ucb_c_min': 0.0, 'ucb_tau': 10.0, 'stagnation_k': 10, 'boost_gamma': 5.0, 'ucb_c_max': None, 'boost_cooldown': 10}

== tsp500 schedule == full=128 subset(step=8)=16 k_sparse=50 n_ants=20 t_aco=[1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]

[schedule cfg] {'ucb_c': 1.6, 'ucb_schedule': 'exp', 'ucb_c_min': 0.0, 'ucb_tau': 10.0, 'stagnation_k': 10, 'boost_gamma': 5.0, 'ucb_c_max': None, 'boost_cooldown': 10}
[schedule] duration=723.50s
  T=  1  avg_cost=21.084288
  T= 10  avg_cost=20.243324
  T= 20  avg_cost=19.703075
  T= 30  avg_cost=19.365173
  T= 40  avg_cost=19.256922
  T= 50  avg_cost=19.063517
  T= 60  avg_cost=18.970282
  T= 70  avg_cost=18.942638
  T= 80  avg_cost=18.806444
  T= 90  avg_cost=18.787655
  T=100  avg_cost=18.773289
  T=110  avg_cost=18.762329
  T=120  avg_cost=18.741222
  T=130  avg_cost=18.724747
  T=140  avg_cost=18.724747
  T=150  a

In [4]:
with tee_stdout(COMPARE_LOG_DIR / 'compare_tsp100_baseline.log', 'w'):
    res_bl_tsp100 = run_baseline_only('tsp100', subsample_step=8)

== tsp100 baseline == full=1280 subset(step=8)=160 k_sparse=20 n_ants=20 t_aco=[1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150]

[baseline c=0] duration=1234.90s
  T=  1  avg_cost=8.998399
  T= 10  avg_cost=8.481752
  T= 20  avg_cost=8.369178
  T= 30  avg_cost=8.316936
  T= 40  avg_cost=8.293048
  T= 50  avg_cost=8.274829
  T= 60  avg_cost=8.253468
  T= 70  avg_cost=8.242595
  T= 80  avg_cost=8.234246
  T= 90  avg_cost=8.226479
  T=100  avg_cost=8.219060
  T=110  avg_cost=8.212718
  T=120  avg_cost=8.208442
  T=130  avg_cost=8.201945
  T=140  avg_cost=8.199124
  T=150  avg_cost=8.193865


In [5]:
with tee_stdout(COMPARE_LOG_DIR / 'compare_tsp100_schedule.log', 'a'):
    print_schedule_run_banner('tsp100')
    res_sc_tsp100 = run_schedule_only('tsp100', subsample_step=8)


2026-05-04T21:42:11.285794  tsp100
schedule={'ucb_c': 1.4, 'ucb_schedule': 'exp', 'ucb_c_min': 0.0, 'ucb_tau': 10.0, 'stagnation_k': 10, 'boost_gamma': 5.0, 'ucb_c_max': None, 'boost_cooldown': 10}

== tsp100 schedule == full=1280 subset(step=8)=160 k_sparse=20 n_ants=20 t_aco=[1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150]

[schedule cfg] {'ucb_c': 1.4, 'ucb_schedule': 'exp', 'ucb_c_min': 0.0, 'ucb_tau': 10.0, 'stagnation_k': 10, 'boost_gamma': 5.0, 'ucb_c_max': None, 'boost_cooldown': 10}
[schedule] duration=1258.15s
  T=  1  avg_cost=9.051713
  T= 10  avg_cost=8.580219
  T= 20  avg_cost=8.429511
  T= 30  avg_cost=8.353499
  T= 40  avg_cost=8.316942
  T= 50  avg_cost=8.289991
  T= 60  avg_cost=8.268685
  T= 70  avg_cost=8.249085
  T= 80  avg_cost=8.237238
  T= 90  avg_cost=8.229205
  T=100  avg_cost=8.223836
  T=110  avg_cost=8.214747
  T=120  avg_cost=8.207899
  T=130  avg_cost=8.204639
  T=140  avg_cost=8.197956
  T=150  avg_cost=8.193715


### 实验一（可选）：delta

三组 baseline 与 schedule 均已运行后，可单独执行下一单元格查看差值（无需重跑推理）。

In [ ]:
print('[tsp20]')
print_delta(T_ACO, res_bl_tsp20['avg'], res_sc_tsp20['avg'])
print('\n[tsp500 @ subset ::8]')
print_delta(T_ACO, res_bl_tsp500['avg'], res_sc_tsp500['avg'])
print('\n[tsp100]')
print_delta(T_ACO, res_bl_tsp100['avg'], res_sc_tsp100['avg'])